# A preconditioned quantum linear solver for periodic two-phase heat conduction

The three listings of Section 4 of the paper, with the output printed there. They go from the package used as a library, through the gate-level isometry, to the whole inverse as one circuit.

Install the package first, from the repository root: `pip install -r requirements.txt`.


## Listing 1. The inverse on the square at $N = 256$, and its resources at $N = 1024$

The simplest use of the package. We build the square cell at $N = 256$ and $\rho = 10$, compute the extreme eigenvalues of $\tilde{M}$, apply the inverse at $\epsilon_p = 10^{-6}$ to the source of the cell problem for the macroscopic gradient $(1, 0)$, and then compute the resources of the inverse at $N = 1024$. Each quantity is a single function call. Here $q_t(\tilde{M})$ is applied as a matrix polynomial, without gates: the condition number is $\rho$ and the energy error stays within $\epsilon_p$. Runs in a few seconds.


In [1]:
from quantumpoissonhomogenization import (Cell, extreme_eigs, inverse_error,
                                          inverse_resources)

cell = Cell("square", N=256, rho=10.0)             # 65,536 unknowns
lo, hi = extreme_eigs(cell)                        # extreme eigenvalues of Mtilde
err, d = inverse_error(cell, eps_p=1e-6)           # q_t(Mtilde), matrix-free
print(f"kappa(Mtilde)    = {hi / lo:.6f}")
print(f"degree d_t       = {d}")
print(f"energy error     = {err:.2e}")

v = inverse_resources("square", N=1024, rho=10.0, eps_p=1e-6)
print(f"F                = {v['F']}")
print(f"Toffoli / query  = {v['toffoli_per_query']:,}")
print(f"Toffoli, inverse = {v['toffoli_total']:.2e}")
print(f"qubits           = {v['qubits']:,}")


kappa(Mtilde)    = 10.000000
degree d_t       = 145
energy error     = 5.03e-07
F                = 45
Toffoli / query  = 88,926
Toffoli, inverse = 1.29e+07
qubits           = 5,699


## Listing 2. The gate-level $\mathcal{U}_V$ at $N = 16$ and $F = 24$, checked against $\hat{V}(k)$ at every wavenumber

One level down, to the circuit itself. We build $\mathcal{U}_V$ at $N = 16$ and $F = 24$ fractional bits, simulate it gate by gate for all $256$ wavenumbers, and compare the vector that $\mathcal{Q}(k)$ prepares with the closed-form $\hat{V}(k)$. The checks also confirm that the work register comes back empty and the wavenumber inputs are restored, which is what makes the circuit reusable inside $\mathcal{U}_M$. Runs in a few seconds.


In [2]:
import numpy as np
from quantumpoissonhomogenization import IsometryCircuit, vhat_exact

circ = IsometryCircuit(n=4, F=24)          # N = 16, F = 24 fractional bits
Q, checks = circ.simulate()                # Q(k)|0>, gate by gate, every k
N = circ.N
err = max(np.linalg.norm(Q[k1 * N + k2, :8, 0] - vhat_exact(N, k1, k2))
          for k1 in range(N) for k2 in range(N))
r = circ.resources()

print(f"wavenumbers      = {N * N}")
print(f"work register    = {checks['work_bits_left_set']} bits left set")
print(f"inputs restored  = {checks['inputs_restored']}")
print(f"max |error|      = {err:.1e}")
print(f"Toffoli gates    = {r['toffoli_total']:,}")
print(f"qubits           = {r['wires']:,}")


wavenumbers      = 256
work register    = 0 bits left set
inputs restored  = True
max |error|      = 1.1e-06
Toffoli gates    = 13,349
qubits           = 2,051


## Listing 3. The entire inverse as one Qiskit circuit at $N = 4$, simulated by state vector

The whole algorithm end to end. We build the inverse, from the right-hand side to the output state, as one Qiskit circuit for the square at $N = 4$, $\rho = 10$ and $\epsilon_p = 10^{-3}$, simulate it by state vector, and compare the output with the finite element solution. The block of $\mathcal{U}_M$ matches $\tilde{M}$ to machine precision and the energy error is below $\epsilon_p$; the success probability is the price of postselection. Runs in about $15$ seconds.


In [3]:
from quantumpoissonhomogenization.inversion import qiskit_inverse

r = qiskit_inverse(N=4, rho=10.0, eps_p=1e-3)    # square cell, 16 unknowns
print(f"degree d_t       = {r['d']}")
print(f"qubits           = {r['qubits']}")
print(f"CX gates         = {r['inverse'][1]:,}")
print(f"block of U_M     = {r['err_M']:.1e}")
print(f"energy error     = {r['energy']:.2e}")
print(f"P(success)       = {r['p_success']:.2e}")


degree d_t       = 77
qubits           = 13
CX gates         = 28,183
block of U_M     = 5.9e-14
energy error     = 6.40e-04
P(success)       = 2.44e-03
